Needs a lot of editing and debugging but will serve as a protocol for analysing RECTIFY images 

# Step 0: Preparation

In [1]:
#best to import some libraries in notebook so there are no issues if you don't want to run cells in a specific order
# import torch
import numpy as np #for running adc quantification in the notebook
import dbdicom as db
import napari
import numpy as np
import math
import matplotlib.pyplot as plt
import os
import importlib
#import rectify.rectify_code as rectify_code
#importlib.reload(rectify_code)
import inspect
import sys
from pathlib import Path
import re
%gui qt

Choose where to take scans from and where to output the results

In [2]:
#import rectify.rectify_code as rectify_code
#importlib.reload(rectify_code)
DATAPATH = r"C:\Users\md1spsx\Documents\Data\rectify\source"
BUILD_PATH = r"C:\Users\md1spsx\Documents\Data\rectify\results"
#DATAPATH = r"C:\Users\cmq26nc\neoadj\venvs\ppln-rectify\scans\Patient_02" #change to RECTIFY_XY for each patient
#BUILD_PATH = r"C:\Users\cmq26nc\neoadj\venvs\rectify\results_02\Patient_02_05" #need to change this to a new folder for each run
IMAGEPATH = os.path.join(BUILD_PATH, 'images')
CHECKPOINTSPATH = os.path.join(BUILD_PATH, 'checkpoints')
os.makedirs(IMAGEPATH, exist_ok=True)
os.makedirs(CHECKPOINTSPATH, exist_ok=True)
resultspath = os.path.join(BUILD_PATH, 'dicom')
SUBJ = 'SheffieldSubject'
DATA = [resultspath, SUBJ, 'Data']
MAPS = [resultspath, SUBJ, 'Maps']
MASKS = [resultspath, SUBJ, 'Masks']

Check environment path to ensure smooth operation

In [ ]:
print(sys.executable)
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate the project root (pyproject.toml) from the notebook directory.")
source_path = str(project_root / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)


Display file names

In [6]:
for series in db.series(DATAPATH):
    print(series)
#important to remember dce data is stored in the disco file

Reading DICOM folder:   0%|          | 0/15302 [00:00<?, ?it/s]

Reading DICOM folder: 100%|██████████| 15302/15302 [00:23<00:00, 653.10it/s]


['C:\\Users\\md1spsx\\Documents\\Data\\rectify\\source', 'RECTIFY_002_V1', ('POLARIS_RECTIFY', 0), ('PERFUSION (flip 30)', 0)]
['C:\\Users\\md1spsx\\Documents\\Data\\rectify\\source', 'RECTIFY_002_V1', ('POLARIS_RECTIFY', 0), ('PERFUSION (flip 4)', 0)]
['C:\\Users\\md1spsx\\Documents\\Data\\rectify\\source', 'RECTIFY_002_V1', ('POLARIS_RECTIFY', 0), ('PERFUSION (flip 2)', 0)]
['C:\\Users\\md1spsx\\Documents\\Data\\rectify\\source', 'RECTIFY_002_V1', ('POLARIS_RECTIFY', 0), ('PERFUSION (flip 10)', 0)]
['C:\\Users\\md1spsx\\Documents\\Data\\rectify\\source', 'RECTIFY_002_V1', ('POLARIS_RECTIFY', 0), ('PERFUSION (flip 20)', 0)]
['C:\\Users\\md1spsx\\Documents\\Data\\rectify\\source', 'RECTIFY_002_V1', ('POLARIS_RECTIFY', 0), ('PERFUSION (flip 30)', 1)]
['C:\\Users\\md1spsx\\Documents\\Data\\rectify\\source', 'RECTIFY_002_V1', ('POLARIS_RECTIFY', 0), ('PERFUSION DISCO', 0)]


# Step 1: Data cleaning

Harmonize the DICOM data, saving the results in the build folder

Similar procedure to lung case

In [4]:
import importlib
import rectify.rectify_code as rectify_code
importlib.reload(rectify_code) #importing these again down here since it didn't work in the top cell

rectify_code.harmonize_dce_data(DATAPATH, DATA)
#rectify_code.harmonize_vfa_data(DATAPATH, DATA)
#runs in about 15-20 minutes (hardware dependent, all estimates in this notebook are based on intel core i7 desktop)
#added lines in rectify_code.py to give both the dicom SeriesDescription (i.e. what is shown in the copying series sections) but also the source folder
#done to check it would still work with minor file naming incosistencies e.g flip_2_ vs flip4_ 

Reading DICOM folder: 0it [00:00, ?it/s]


TypeError: stat: path should be string, bytes, os.PathLike or integer, not list

In [5]:
from rectify.rectify_code import view_dce_notebook

view_dce_notebook(DATA)
# ~2.5 mins it will keep running after the volume is 100% read just give it a minute and napari will open. 
# #Close napari to continue running the notebook if run all was selected.
#Be careful if napari crashes the kernel crashes and with perfusion disco it crashes a lot

Reading volume..: 100%|██████████| 15000/15000 [00:32<00:00, 464.97it/s]


# Step 2: Map Computaation

Compute some descriptive maps that can be used as a basis for segmentation and eventually calculate T1 and S0

ADC maps calculated using DWI images (just axial for patient 1)

In [ ]:
from rectify.rectify_code import compute_descriptive_maps

compute_descriptive_maps(DATA, MAPS)
# ~3mins
# only works with dce data

Create an ADC map from the axial DWI series with T2 weighted images, will add coronal and saggital if later cases have DWI in those planes

b-values taken from DICOM folders

In [ ]:
from rectify.rectify_code import adc_map

adc_map(DATAPATH, MAPS)

In [ ]:
from rectify.rectify_code import view_adc_notebook

view_adc_notebook(MAPS)
# dark spots at tumour sites due to limited browmian motion in tumour volume, bright spots are only tumour sites in dwi images

In [ ]:
adc = db.volume(MAPS + ['ADC']).values
valid = np.isfinite(adc) & (adc > 0)
if not np.any(valid):#
    raise ValueError("The ADC map has no positive finite values.")
print(f"Maximum ADC: {(np.max(adc[valid])):.5g} mm^2/s")
print(f"Top 1% ADC Values: {np.percentile(adc[valid], 99):.5g} mm^2/s")
print(f"Top 10% ADC Values: {np.percentile(adc[valid], 90):.5g}mm^2/s")
print(f"Mean ADC Value: {np.percentile(adc[valid], 50):.5g} mm^2/s") #(ADC of rectal wall typically 0.0018-0.001405)
print(f"Bottom 10% ADC Value: {np.percentile(adc[valid], 50):.5g} mm^2/s")
print(f"Bottom 1% Value: {np.percentile(adc[valid], 50):.5g} mm^2/s")
print(f"Minimum ADC: {(np.min(adc[valid])):.5g}  mm^2/s")

# Step 3: Image Segmentation

Colon can be automatic but rectum will have to be done manually

Automatic segmentation will have to be completely given up for manual

In [ ]:
from rectify.rectify_code import manually_segment_ax_t2

manually_segment_ax_t2(DATAPATH, BUILD_PATH)

In [ ]:
from rectify.rectify_code import manually_segment_vfa

manually_segment_vfa(MAPS, MASKS, BUILD_PATH, reference_flip_angle=2)

In [ ]:
#from rectify.rectify_code import autosegment_colorectal

#autosegment_colorectal(MAPS, MASKS)
# ~5 mins 
#this is the very loud step
#should skip for colorectal and segment manually

Check the masks and edit interactively, if needed. Almost certainly needed for rectum due to lack of total_mr map, all needed actually

VFA segmentation uses the motion-corrected flip-angle series, so do not run it at this stage. After the VFA motion-correction cell in Step 4, run the **Shared manual mask for motion-corrected VFA** cell below to create one mask aligned to all flip angles.

In [ ]:
#from rectify.rectify_code import edit_autosegmented_colorectal

#edit_autosegmented_colorectal(MAPS, MASKS)

#It seems like all segmentation in this project will need to be manually done, look into methods for this.

In [ ]:
from  rectify.rectify_code import draw_arterial_inputs_notebook

draw_arterial_inputs_notebook(MAPS, MASKS)
# Close when finished to continue

# Step 4: Breathing motion correction

Motion-correct the VFA data for signal calibration

In [ ]:
from rectify.rectify_code import vfa_motion_correction

vfa_motion_correction(DATA, MAPS)
# ~2 hours

Check the motion-corrected VFA data

In [ ]:
#manual vfa
vfa_manual_mask = rectify_code.manually_segment_vfa(
    MAPS, MASKS, BUILD_PATH, reference_flip_angle=2
)

In [ ]:
from rectify.rectify_code import view_vfa_notebook

view_vfa_notebook(MAPS)
# Gives a pretty good output

Perform DCE motion correction

*Note* this is actually a dummy function for this demo which just creates a copy. Motion correction takes a long time and these colorectal scans don't really need it as the abdominal motion is minimal, so in this case it can be safely skipped.

In [ ]:
#from rectify.rectify_code import dce_motion_correction #skip if dce images not present

#dce_motion_correction(DATA, MAPS)

# Step 5: Signal calibration

Fit the VFA data to derive R1 and S0 maps


In [ ]:
from rectify.rectify_code import vfa_fit

vfa_fit(MAPS)
#kernel crashed halfway through check if this will happen again, have to restart the script and see if it happens
#kernel also sometimes dies in earlier steps, is this due to very large number of images?

In [ ]:
import rectify.rectify_code as rc

r1_vol = rc.db.volume(MAPS + ['R1_vfa']) #load fitted volumes and identify location and series
s0_vol = rc.db.volume(MAPS + ['S0_vfa'])

viewer = rc.napari.Viewer()
viewer.add_image(r1_vol.values.T, name='R1_vfa', colormap='viridis')
viewer.add_image(s0_vol.values.T, name='S0_vfa', colormap='magma')
viewer

#r1_vol = rc.db.volume(MAPS + ['R1_vfa'])
#s0_vol = rc.db.volume(MAPS + ['S0_vfa'])

#r1 = r1_vol.values
#s0 = s0_vol.values

#n_slices = r1.shape[2]
#n_rows = math.ceil(n_slices / 2)

# Keep the color scale consistent across slices of each map.
#r1_min, r1_max = np.nanmin(r1), np.nanmax(r1)
#s0_min, s0_max = np.nanmin(s0), np.nanmax(s0)

#fig, axes = plt.subplots(
 #   n_rows, 4,
  #  figsize=(16, 4 * n_rows),
   # squeeze=False,
#)

for i in range(n_slices):
    row = i // 2
    first_col = (i % 2) * 2

    r1_image = axes[row, first_col].imshow(
        r1[:, :, i].T, cmap='viridis', origin='lower',
        vmin=r1_min, vmax=r1_max,
    )
    axes[row, first_col].set_title(f'R1 — slice {i + 1}')
    axes[row, first_col].axis('off')
    fig.colorbar(r1_image, ax=axes[row, first_col])

    s0_image = axes[row, first_col + 1].imshow(
        s0[:, :, i].T, cmap='viridis', origin='lower',
        vmin=s0_min, vmax=s0_max,
    )
    axes[row, first_col + 1].set_title(f'S0 — slice {i + 1}')
    axes[row, first_col + 1].axis('off')
    fig.colorbar(s0_image, ax=axes[row, first_col + 1])

# Hide unused panels if there is an odd number of slices.
for col in range((n_slices % 2) * 2, 4):
    if n_slices % 2:
        axes[-1, col].axis('off')

fig.tight_layout()
plt.savefig(os.path.join(IMAGEPATH, 'R1_S0_slices.png'), dpi=300)
plt.show()

In [ ]:
# Use robust display limits to check a few high outliers do not dominate the color scale on the previous maps (very small bright spots on the S0 map with a scale up to 700,000)
def positive_percentile_vmax(array, percentile=99):
    positive_values = array[np.isfinite(array) & (array > 0)]
    if positive_values.size == 0:
        raise ValueError("Map contains no finite positive values to display.")
    return float(np.percentile(positive_values, percentile))


n_slices = r1.shape[2]
n_rows = math.ceil(n_slices / 2)

# Keep the robust color scale consistent across slices of each map.
r1_min, r1_max = 0, positive_percentile_vmax(r1)
s0_min, s0_max = 0, positive_percentile_vmax(s0)

fig, axes = plt.subplots(
    n_rows, 4,
    figsize=(16, 4 * n_rows),
    squeeze=False,
)

for i in range(n_slices):
    row = i // 2
    first_col = (i % 2) * 2

    r1_image = axes[row, first_col].imshow(
        r1[:, :, i].T, cmap='viridis', origin='lower',
        vmin=r1_min, vmax=r1_max,
    )
    axes[row, first_col].set_title(
        f'R1 - slice {i + 1} (99th percentile scale)'
    )
    axes[row, first_col].axis('off')
    fig.colorbar(r1_image, ax=axes[row, first_col])

    s0_image = axes[row, first_col + 1].imshow(
        s0[:, :, i].T, cmap='viridis', origin='lower',
        vmin=s0_min, vmax=s0_max,
    )
    axes[row, first_col + 1].set_title(
        f'S0 - slice {i + 1} (99th percentile scale)'
    )
    axes[row, first_col + 1].axis('off')
    fig.colorbar(s0_image, ax=axes[row, first_col + 1])

# Hide unused panels if there is an odd number of slices.
for col in range((n_slices % 2) * 2, 4):
    if n_slices % 2:
        axes[-1, col].axis('off')

fig.tight_layout()
plt.savefig(os.path.join(IMAGEPATH, 'R1_S0_slices_robust.png'), dpi=300)
plt.show()

Align VFA and DCE data, while computing scaling factors for the MR signal intensities (arbitrary units).

In [ ]:
from rectify.rectify_code import dce_motion_correction #might actually need to use this
dce_motion_correction(DATA, MAPS)

#dce motion correction doesn't take nearly as long as I thought: less than 10 minutes

from rectify.rectify_code import align_vfa_with_dce

align_vfa_with_dce(MAPS, MASKS)
#need to make edits to allow alignment to work without totalsegmentator

# Step 6: Parameter reconstruction

Now we have everything in place to model the data and derive the parameters of interest.

Model-based colon analysis

In [ ]:
from rectify.rectify_code import roi_analysis_model_based

roi_analysis_model_based(MAPS, MASKS, IMAGEPATH)

In [ ]:
from rectify.rectify_code import view_png

png = os.path.join(IMAGEPATH, f'model_based_colon_edited_rectify01.png')
view_png(png)


Model-free colon analysis

In [ ]:
from rectify.rectify_code import roi_analysis_model_free

roi_analysis_model_free(MAPS, MASKS, IMAGEPATH)

In [ ]:
from rectify.rectify_code import view_png

png = os.path.join(IMAGEPATH, f'model_free_colon_edited_rectify01.png')
view_png(png)


Pixel-by-pixel model-free analysis

In [ ]:
from rectify.rectify_code import pixel_analysis_model_free

pixel_analysis_model_free(MAPS, MASKS, IMAGEPATH)

In [ ]:
from rectify.rectify_code import view_png

png = os.path.join(IMAGEPATH, f'Fb_model_free_rectify01.png')
view_png(png)
png = os.path.join(IMAGEPATH, f'vb_model_free_rectify01.png')
view_png(png)

Pixel-by-pixel model-based analysis

In [ ]:
from rectify.rectify_code import pixel_analysis_model_based

pixel_analysis_model_based(MAPS, MASKS, IMAGEPATH, CHECKPOINTSPATH)

In [ ]:
from rectify.rectify_code import view_png

png = os.path.join(IMAGEPATH, f'Fb_model_based_rectify01.png')
view_png(png)
png = os.path.join(IMAGEPATH, f'vb_model_based_rectify01.png')
view_png(png)

# Step 7: Export imaging biomarkers

In [ ]:
from rectify.rectify_code import pixel_averages_model_based

pixel_averages_model_based(MAPS, MASKS, SUBJ, BUILD_PATH)